In [3]:
# Install necessary libraries
!pip install datasets transformers -q

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
import os

In [4]:
# Disable W&B
os.environ["WANDB_DISABLED"] = "true"

# Load SNIPS dataset
dataset = load_dataset("snips_built_in_intents")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 328
    })
})


In [5]:
# Tokenize the dataset
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [6]:
# Split dataset
train_test_valid_split = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=42)
valid_test_split = train_test_valid_split["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = train_test_valid_split["train"]
eval_dataset = valid_test_split["train"]
test_dataset = valid_test_split["test"]

# Print dataset details
print(f"Train: {len(train_dataset)} samples")
print(f"Validation: {len(eval_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

Train: 262 samples
Validation: 33 samples
Test: 33 samples


In [8]:
# Intent Classification Setup
intent_labels = dataset["train"].features["label"].names
num_labels = len(intent_labels)

# Load intent classification model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=num_labels
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train and Evaluate Intent Model
trainer.train()
results = trainer.evaluate(test_dataset)
print("Intent Classification Results:", results)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.073400,1.965021,0.393939,0.162656,0.393939,0.228195
2,1.662200,1.617466,0.515152,0.451659,0.515152,0.407071
3,1.338300,1.294578,0.636364,0.545455,0.636364,0.545455
4,1.083500,1.072491,0.848485,0.856749,0.848485,0.833237
5,0.926200,0.968784,0.878788,0.927273,0.878788,0.881663
6,0.833500,0.920138,0.878788,0.927273,0.878788,0.881663


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

Intent Classification Results: {'eval_loss': 0.8707167506217957, 'eval_accuracy': 0.8484848484848485, 'eval_precision': 0.7859110586383314, 'eval_recall': 0.8484848484848485, 'eval_f1': 0.808080808080808, 'eval_runtime': 0.99, 'eval_samples_per_second': 33.335, 'eval_steps_per_second': 5.051, 'epoch': 6.0}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
import torch

# Check if GPU is available and move the model to the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Prepare input and move it to the same device as the model
sample_text = "Find me a 10 km scenic route"
inputs = tokenizer(sample_text, return_tensors="pt", padding=True, truncation=True)
inputs = {key: value.to(device) for key, value in inputs.items()}  # Move inputs to device

# Perform forward pass
outputs = model(**inputs)

# Get predicted label
predicted_label = intent_labels[outputs.logits.argmax().item()]
print(f"Predicted Intent: {predicted_label}")

Predicted Intent: GetDirections
